# 2-clean&filter

## 2.1 pré-nettoyage, pré-filtrage et pré-recodages

In [ ]:
import pandas as pd
import re

# Charger le df concaténé des deux législatures
df = pd.read_csv(
    "../data/interim/interventions_regroupees.csv",  # interventions_regroupees ou extract_15_16_concat
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

print("Shape du df chargé : ", df.shape)

# ==============================
# Pré-nettoyage et pré-filtrage
# ==============================
"""
nb : précision choix 
- exclusion président.e :
role_debat n'est pas toujours bien identifié, utiliser aussi nom_orateur
(avant de le recoder/nettoyer car sinon risque perte par remplacement)
"""

# ===========================================================
# FILTRES RAPIDES
# ===========================================================

# Exclure les prises de parole de "Mme la présidente" et "M. le président"
df = df[~df["nom_orateur"].str.strip().isin(["M. le président", "Mme la présidente"])]
# Exclure les éléments restants en roledebat == president
df = df[~df["roledebat"].str.strip().isin(["president"])]

# Exclure les lignes pour lesquelles on n'a pas d'info orateurs (pas exploitable ici)
df = df[~(df["id_acteur"].isna() & df["nom_orateur"].isna() & df["id_orateur"].isna())]

# Ne garder que le code style NORMAL
df = df[df["code_style"] == "NORMAL"]

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
# (normalement géré dans la section fusion des interventions interrompues)
# (mais si des gens veulent s'en passer le faire ici)
df["code_parole"] = df["code_parole"].fillna("non_précisé")

# Garder une trace de la longueur des interventions brutes
# (nb : varie selon qu'on groupe ou non les interventions interrompues en amont)
df["len_texte_brut"] = df["texte"].str.len()

print("Shape du df après pré-filtrage : ", df.shape)

# ========== Gestion codes grammaires ==========
# virer les codes grammaires inutiles qui restent après premiers filtres
# (peut varier selon les choix de filtrage en amont)
exclusion_code_grammaire = [
    "OUV_SEAN_2_1",
    "FUSION",
    "FIN_SEAN_2_1",
    "DISC_ARTICLES_3_9_1",  # interv président.es mal identifiées
    "DISC_ARTICLES_3_1",
    "ANN_SCR_AUTRE_1_0",
    # Doute, probablement pas utile de virer :
    # "ODJ_APPEL_DISCUSSION"
    # "DISC_ARTICLES_1_30"
]

n_avant = len(df)

mask_excl_code_grammaire = df["code_grammaire"].isin(exclusion_code_grammaire)
print(f"Lignes à exclure (code_grammaire) : {mask_excl_code_grammaire.sum()}")

df = df[~mask_excl_code_grammaire]

print(f"Shape après exclusion code_grammaire : {df.shape})")


# ===========================================================
# NETTOYAGE DES TEXTES PUIS EXCLUSION DES VIDES ET PARASITES
# ===========================================================

# ========== Nettoyer les textes et supprimer les lignes vides ==========


# nettoyage basique du texte
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes (nécessaire pour regex)
    texte = texte.replace("’", "'")
    return texte


df["texte_brut"] = df["texte"]  # garder une version brute du texte
df["texte"] = df["texte"].apply(nettoyer_texte)

# supprimer les lignes où "texte" est manquant ou vide
mask_texte_vide = df["texte"].isna() | (df["texte"] == "")
print(f"Lignes supprimées pour texte vide : {mask_texte_vide.sum()}")
df = df[~mask_texte_vide]


# ========== Gestion textes parasites ==========

# Varie selon les choix de filtrage en amont (sur les président.es)
# mais il peut y avoir quelques cas de textes parasites
# En fait juste quelques cas très spécifiques.
# Mais autant avoir une fonction si on devait généraliser à d'autres données

# cf les spécifiques :
# "……………………………………………………………."
# "………………………………………………………………………………………"
# "------------------Cette partie de la séance est en cours de finalisation---------------------------------------------"  #
# "------------------Cette partie de la séance est en cours de finalisation---------------------------------------------Madame la ministre, (blablablabla)"


# identification textes parasites
def est_texte_parasite(texte):
    if not isinstance(texte, str):
        return False
    nettoyé = re.sub(
        r"Cette partie de la séance est en cours de finalisation", "", texte
    )
    nettoyé = re.sub(r"[-–—.…\s]+", "", nettoyé)
    return nettoyé == ""


mask_parasite = df["texte"].apply(est_texte_parasite)
print(f"Lignes parasites supprimées : {mask_parasite.sum()}")
df = df[~mask_parasite]

print("Shape du df après nettoyage et exclusion des textes : ", df.shape)


# ===========================================================
# FUSION DES IDENTIFIANTS D'ACTEURS
# (et possible correction manuelle des erreurs)
# (ou par recalcul plus bas)
# TODO: choix si solution automatique ou manuelle
# ===========================================================

# ========== Fusion id_acteur et id_orateur ==========
"""
NOTE:
- id_acteur vs id_orateur :
certains cas (~3000) ont un id_orateur plus précis (un code PA) que id_acteur qui a PA0
Mais en fait ce sont presque 100% des interruptions avec souvent plusieurs locuteurs.
id_orateur en renvoie (mal) un seul -> on préfère garder le PA0 (neutre)
"""

# Stabiliser le id_orateur pour être au format AN
df["id_orateur"] = "PA" + df["id_orateur"]
# Remplacer les valeurs manquantes de id_acteur par id_orateur quand disponible
df["id_acteur_originel"] = df["id_acteur"]  # garder une trace
df["id_acteur"] = df["id_acteur"].combine_first(df["id_orateur"])


# # ========== Possible correction manuelle des erreurs id_acteur =============

# TODO: choix si solution automatique ou manuelle
# (peut changer si on modifie les choix filtrage en amont)
# # ICI POUR L'INSTANT GÉRÉ PAR FONCTION IDENTIFICATION + RECALCUL NOM
# # UN PEU PLUS BAS DANS LE NOTEBOOK

# # Cas identifiés où id_acteur != id_orateur ET les noms diffèrent
# # -> on considère que le texte (et donc le nom inscrit) du CR fait foi
# # -> l'id_orateur est le bon, on l'utilise pour écraser id_acteur
# # (liste établie a posteriori via diagnostic id_pb, figée ici pour éviter de recalculer)

# ID_SYCERON_A_CORRIGER = {
#     3024324,  # "M. Hadrien Clouet"  -> identifié sous PA Tavel (PA794166)
#     3048161,  # "Mme Lisa Belluco"   -> identifié sous PA Trouvé (PA795164)
#     3180585,  # "M. Benjamin Lucas"  -> identifié sous PA Guedj (PA1567)
#     3182770,  # "M. Sacha Houlié"    -> identifié sous PA Dupond-Moretti (PA773443)
#     3204694,  # "M. Matthias Tavel"  -> identifié sous PA Trouvé (PA795164)
#     3205482,  # "M. Bruno Millienne" -> identifié sous PA Croizier (PA793716)
#     3243209,  # "M. Guillaume Kasbarian" -> identifié sous PA Grégoire (PA721764)
#     3260505,  # "M. Jean-René Cazeneuve" -> identifié sous PA Cazenave (PA793940)
#     3275233,  # "M. Frédéric Cabrolier"  -> identifié sous PA Minot (PA720630)
#     3286452,  # "M. Jean-René Cazeneuve" -> identifié sous PA Cazenave (PA793940)
#     3315478,  # "M. Hadrien Ghomi"   -> identifié sous PA Pellerin (PA795926)
#     3321985,  # "M. Manuel Bompard"  -> identifié sous PA Lecoq (PA335612)
#     3378626,  # "M. André Chassaigne"-> identifié sous PA Molac (PA607619)
#     3471389,  # "M. Grégoire de Fournas" -> identifié sous PA Prud'homme (PA719578)
# }

# mask_syceron_pb = df["id_syceron"].isin(ID_SYCERON_A_CORRIGER)
# df.loc[mask_syceron_pb, "id_acteur"] = df.loc[mask_syceron_pb, "id_orateur"]
# # ========== FIN Correction manuelle si nécessaire =============


# ===========================================================
# RÉCUPÉRER ET NETTOYER LES NOMS LES PLUS FRÉQUENTS
# POUR CHAQUE ID_ACTEUR SAUF PA0 ET LES ID_ACTEUR MANQUANTS'
# (et possible corrections automatique a posteriori des erreurs id_acteur)
# TODO : choisir si solution automatique ou manuelle
# ===========================================================

# ========== Recoder par noms les plus fréquents ==========

# Nom le plus fréquent
most_frequent_name = df.groupby("id_acteur")["nom_orateur"].agg(
    lambda x: x.dropna().mode().iloc[0] if x.dropna().size > 0 else None
)  # version plus stable que value_counts().idxmax() en cas d'ex-aequo


# Renvoyer le nom le plus fréquent sauf si id_acteur == PA0 ou id_acteur est manquant
# NOTE : voir imite de la fonction en description


def get_most_frequent_name(row):
    """
    Récupération de la forme la plus fréquente du nom,
    uniquement pour acteurs différents de PA0.
    if PA0 : nom brut, else : nom le plus fréquent pour cet id.
    /!\ Limite de la fonction :
    - 1. invisibilise les rares cas d'interventions mal identifiées
    par leur PA, mais qui ont le bon nom (ici le nom majoritaire sera renvoyé)
    - 2. pourrait poser problème si on gardait les M/Mme président.e
    (qui pourraient être majoritaires plutôt que le nom de la personne)
    """
    if row["id_acteur"] == "PA0" or pd.isna(row["id_acteur"]):
        return row["nom_orateur"]
    return most_frequent_name.get(row["id_acteur"], row["nom_orateur"])


df["nom_orateur_clean"] = df.apply(get_most_frequent_name, axis=1)


# ========== Nettoyer les noms d'orateurs ==========


def nettoyer_nom(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les virgules
    texte = texte.replace(",", " ")
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour les apostrophes
    texte = texte.replace("’", "'")
    return texte


df["nom_orateur_clean"] = df["nom_orateur_clean"].apply(nettoyer_nom)


# # ========== Correction automatique des erreurs id_acteur =============
# TODO: choix si solution automatique ou manuelle

"""
NOTE : nb traçabilité :
Cas problématiques id_acteur != id_orateur & nom_orateur != nom_orateur_clean
ie : Si id_acteur != id_orateur mais que les noms sont similaires,
c'est que l'erreur porte sur l'id_orateur et ça nous gène pas (pas celui utilisé ensuite).

nb : Dans notre cas, l'égalité sur la base des noms nettoyés suffit
(après vérification manuelle de tous les cas)
Mais en réalité, il faudrait sans doute un truc plus tolérant
si on voulait généraliser à d'autres données.
ex : fuzzy fuzz ou virer les accents, etc.
"""

# Mask des cas problématiques id_acteur vs id_orateur et noms différents
mask_pb_id = (
    df["id_acteur"].notna()
    & df["id_orateur"].notna()
    & (df["id_acteur"] != "PA0")
    & (df["id_orateur"] != "PA0")
    & (df["id_acteur"] != df["id_orateur"])
    & (
        df["nom_orateur"].apply(nettoyer_nom) != df["nom_orateur_clean"]
    )  # nettoyer pour commensurabilité
)
# cols pour affichage diagnostic
cols_check = [
    "id_acteur",
    "id_orateur",
    "nom_orateur",
    "nom_orateur_clean",
    "id_syceron",
]

print("\nSituations problématiques id_acteur vs id_orateur avant correction :\n")
print(df.loc[mask_pb_id, cols_check].sort_values("id_syceron").to_string(index=False))

# Correction : id_acteur <- id_orateur
df.loc[mask_pb_id, "id_acteur"] = df.loc[mask_pb_id, "id_orateur"]
print(f"\nLignes corrigées dans df : {mask_pb_id.sum()}")

# Recalculer le nom recodé après correction des identifiants acteurs.
# = réactualiser (puisqu'on en a modifié, pourrait changer le plus fréquent)
most_frequent_name = df.groupby("id_acteur")["nom_orateur"].agg(
    lambda x: x.dropna().mode().iloc[0] if x.dropna().size > 0 else None
)
# et ré-appliquer la récup (= la fonction change pas)
df["nom_orateur_clean"] = df.apply(get_most_frequent_name, axis=1).apply(nettoyer_nom)

print("\nSituations après correction :\n")
print(
    df.loc[mask_pb_id, cols_check]
    .drop_duplicates("id_syceron")
    .sort_values("id_syceron")
    .to_string(index=False)
)

print("\nShape du df en sortie : ", df.shape)


Shape du df chargé :  (1013513, 33)
Shape du df après pré-filtrage :  (571071, 34)
Lignes à exclure (code_grammaire) : 0
Shape après exclusion code_grammaire : (571071, 34))
Lignes supprimées pour texte vide : 12
Lignes parasites supprimées : 0
Shape du df après nettoyage et exclusion des textes :  (571059, 35)

Situations problématiques id_acteur vs id_orateur avant correction :

id_acteur id_orateur            nom_orateur      nom_orateur_clean  id_syceron
 PA794166   PA793736      M. Hadrien Clouet      M. Matthias Tavel     3024324
 PA795164   PA795362       Mme Lisa Belluco     Mme Aurélie Trouvé     3048161
   PA1567   PA795636      M. Benjamin Lucas        M. Jérôme Guedj     3180585
 PA773443   PA722150        M. Sacha Houlié M. Éric Dupond-Moretti     3182770
 PA795164   PA794166      M. Matthias Tavel     Mme Aurélie Trouvé     3204694
 PA793716   PA721976     M. Bruno Millienne    M. Laurent Croizier     3205482
 PA721764   PA719372 M. Guillaume Kasbarian    Mme Olivia Grégo

#### CAS CONFLITS NOMS

In [8]:
# TODO : les autres repérés dans le identif.csv et ajout_id_acteur.csv (si pas déjà dans identif)
# TODO: syceron 2827575 2827576 2827577 = M. Lionel Tivoli = PA793298


In [9]:
# TODO: MATTHIAS -> nope, LÉO EN COURS
# suite ajout comparaison nom origine vs nom clean = permet repérer erreurs id_acteur (cf cas ministre, etc.)
# DÉSORMAIS CHOISIR SI ON VEUt EN FAIRE UN TRUC OU SI TROP MARGINAL

# dessous :
# 1/ fuzzyfuzz
# 2/ aussi une version avec id_acteur vs id_orateur pour cas évidents de possible pb

In [10]:
# ========== Comparaison nom_orateur brut vs nom_orateur_clean ==========
# Repérer les cas où le nom le plus fréquent assigné à un id_acteur
# diffère du nom brut de l'intervention -> signe possible d'une erreur d'id_acteur
# (ex : ministre ou invité avec un PA qui appartient à un autre)
# Passage par un rapidfuzz pour avoir un score de similarité

from rapidfuzz import fuzz


def normaliser_nom_fuzzy(x):
    if not isinstance(x, str):
        return x
    x = nettoyer_nom(x).lower().strip()
    # retire ponctuation pour éviter de flaguer juste une virgule/point
    x = re.sub(r"[^\w\s'-]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


# Noms normalisés
# pour nom orateur = appliquer aussi le nettoyage standard pour comparabilité
nom_brut_norm = df["nom_orateur"].apply(nettoyer_nom).apply(normaliser_nom_fuzzy)
nom_clean_norm = df["nom_orateur_clean"].apply(normaliser_nom_fuzzy)

# Score fuzzy (0-100)
score_fuzzy = [
    fuzz.token_sort_ratio(a, b) if isinstance(a, str) and isinstance(b, str) else None
    for a, b in zip(nom_brut_norm, nom_clean_norm)
]
df["score_nom_fuzzy"] = score_fuzzy

# Seuil: plus haut = plus strict
seuil_similarite = 96

cols_fuzz = [
    "id_acteur",
    "id_acteur_originel",
    "id_orateur",
    "nom_orateur",
    "nom_orateur_clean",
    "id_syceron",
    "score_nom_fuzzy",
]
mask_nom_diff_significatif = (
    (df["id_acteur"] != "PA0")
    & nom_brut_norm.notna()
    & nom_clean_norm.notna()
    & (nom_brut_norm != nom_clean_norm)
    & (df["score_nom_fuzzy"] < seuil_similarite)
)

deduplication_pb_fuzz = (
    df.loc[mask_nom_diff_significatif, cols_fuzz]
    .value_counts()
    .rename("n")
    .reset_index()
    .sort_values("score_nom_fuzzy")
)

deduplication_pb_fuzz.to_csv("../data/temp/deduplication_pb_fuzz.csv", index=False)
deduplication_pb_fuzz


,id_acteur,id_acteur_originel,id_orateur,nom_orateur,nom_orateur_clean,id_syceron,score_nom_fuzzy,n
1960,PA605991,PA605991,PA605991,Plusieurs députés,Mme Annie Genevard,2746471,17.142857,1
1381,PA1008,PA1008,PA1008,Un député du groupe RN,M. Alain David,2843758,17.142857,1
1968,PA717379,PA717379,PA717379,Un député du groupe RE,M. Sylvain Maillard,3022684,20.000000,1
958,PA721824,PA721824,PA721824,M. Éric Poulliat,M. Hugues Renson,2394546,20.000000,1
631,PA795266,PA795266,PA795266,Un député du groupe SOC,Mme Agnès Carel,2963825,21.052632,1
...,...,...,...,...,...,...,...,...
784,PA721764,PA721764,PA721764,Mme Olivia Gregoire,Mme Olivia Grégoire,2152026,94.736842,1
783,PA721764,PA721764,PA721764,Mme Olivia Gregoire,Mme Olivia Grégoire,2159912,94.736842,1
782,PA721764,PA721764,PA721764,Mme Olivia Gregoire,Mme Olivia Grégoire,2160305,94.736842,1
780,PA721764,PA721764,PA721764,Mme Olivia Gregoire,Mme Olivia Grégoire,2160693,94.736842,1


## 2.2 Match des infos sur les députés (données datan)

### 2.2.1 Match des infos générales

In [11]:
# ==============================
# MATCH DONNÉES DÉPUTÉS
# ==============================

df_deputes = pd.read_csv("../data/raw/id-dep/deputes-historique(datan-datagouv).csv")

# suppression des colonnes non utiles qui introduisent soucis parsing
df_deputes = df_deputes.drop(
    columns=[
        "mail",
        "twitter",
        "facebook",
        "website",
        "active",
        "scoreParticipationSpecialite",
        "datePriseFonction",
        "groupe",
        "naissance",
    ]
)

# ======Fusion des données députés======

print("shape avant fusion:", df.shape)

assert df_deputes["id"].is_unique, "ids du df_deputes non uniques !"

# Merger et virer la col id pour éviter doublon
df = df.merge(
    df_deputes,
    left_on="id_acteur",
    right_on="id",
    how="left",
    suffixes=("", "_dep"),
    validate="many_to_one",  # check if merge keys are unique in right dataset
).drop(columns=["id"])  # supprimer la colonne id du df_deputes

print("shape après fusion données députés:", df.shape)

shape avant fusion: (571059, 38)
shape après fusion données députés: (571059, 55)


### 2.2.2 Match temporel des affiliations

In [12]:
# ======================================================
# AFFILIATION PARTISANE
# Logique suivie :
# 1. récupérer le groupe a date d'intervention si dispo
#   -> var affiliation_mandat_députés
# 2. ajouter les affiliations gouvernementales
#   -> var affiliation_et_gouv
#   - corriger les données manquantes gouv quand entre des bornes gouv meme jour
#   - aviser cas limites = TODO
# 3. fallback sur dernière affiliation connue (groupe/groupeabrev)
#   -> var affiliation_et_gouv complétée
#   - TODO: actuellement ne recupère plus que 3 et peut être qu'il faut pas les corriger
#   - correction des affil NI/RN
# 5. optionnel variable supplémentaire avec uniquement affiliation partisane
#   - écraser affil GOUV par leur rattachement partisan
#   - gérer les cas limites
# ======================================================

In [13]:
# ======================================================
# RECODAGE ET MATCH TEMPOREL DES AFFILIATIONS PARTISANES
# cf. affiliation lors de telle prise de parole
# ======================================================


# ========== Recodage des dénominations de groupes ==========
"""
nb : ici choix de recoder avec les nom des partis,
car ils sont moins sensible aux évolutions marginales de noms,
même si en réalité les groupes parlementaires sont + larges que les partis
et peuvent servir à accueillir des NI d'étiquettes diverses
"""

# Lecture du fichier d'affiliation par périodes
df_affiliation = pd.read_csv(
    "../data/raw/id-dep/datan_affiliations.csv", encoding="latin1", sep=";"
)  # format dégueu

# Recodage des partis pour stabilité temporelle des noms
recodage_affiliation = {
    "RE": "REN",
    "EPR": "REN",
    "LAREM": "REN",
    "MODEM": "DEM",
    "SOC": "SOC-A",
    "NG": "SOC-A",
    "LFI-NUPES": "LFI",
    "FI": "LFI",
    "UDI-AGIR": "UDI",
    "UDI-A-I": "UDI",
    "LC": "UDI",
    "UDI_I": "UDI",
    "UDI-I": "UDI",
    "ECOLO": "ECO",
    "GDR-NUPES": "GDR",
    "LT": "LIOT",
    # Garde pour trace mais pas nécessaire car pas de changement
    # "LIOT": "LIOT",
    # "LR": "LR",
    # "RN": "RN",
    # "MODEM": "MODEM",
    # "LFI": "LFI",
    # "HOR": "HOR",
    # "DEM": "DEM",
}

# Application du recodage des noms de partis au df d'affiliation
df_affiliation["libelleAbrev"] = df_affiliation["libelleAbrev"].astype(str).str.strip()
df_affiliation["parti_recod"] = df_affiliation["libelleAbrev"].replace(
    recodage_affiliation
)

# ========== Match temporel des affiliations ==========

"""
nb : Plutôt qu'un merge foireux, parti sur un lookup ligne à ligne
(= pb des orateurs non députés qui étaient pas présents, etc.)
Le fichier est suffisamment réduit pour que le surplus de calcul soit pas un pb
nb : attention aux bornes temporelles (cf.normalize() pour ignorer l'heure)
"""

# préparation des dates
df["dateSeance_ts"] = pd.to_datetime(
    df["dateSeance"], format="%Y%m%d%H%M%S%f", errors="raise"
)
df_affiliation["dateDebut"] = pd.to_datetime(
    df_affiliation["dateDebut"], errors="raise"
)
df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")
# aviser si jamais besoin un jour de traiter des affiliations en cours
# df_affiliation["dateFin"] = df_affiliation["dateFin"].fillna(pd.Timestamp("2100-01-01"))

# indexer par mpId pour lookup rapide
aff_by_mp = {
    mp: g[["dateDebut", "dateFin", "parti_recod"]].to_dict("records")
    for mp, g in df_affiliation.groupby("mpId")
}


# Fonction de recodage temporel des affiliations
def get_parti_for_row(row):
    """
    Retourne l'affiliation partisane recodée correspondant à la date de séance.

    La fonction :
    - lit `id_acteur` (assimilé à `mpId`) et `dateSeance_ts` sur la ligne ;
    - parcourt les périodes d'affiliation de ce député (si présent dans aff_by_mp);
    - renvoie `parti_recod` si `dateSeance_ts` (normalisée au jour) est comprise
    entre `dateDebut` et `dateFin` (bornes incluses).
    """
    mp = row.get("id_acteur")  # correspond au mpId
    # gérer le cas des orateurs non députés ou autre type intervention
    if pd.isna(mp) or mp not in aff_by_mp:
        return None
    # récupérer le ts de l'intervention
    ts = row.get("dateSeance_ts")
    if pd.isna(ts):
        return None
    # retourner l'affiliation qui colle à la date d'intervention
    for rec in aff_by_mp[mp]:
        # attention : .normalize() pour ignorer l'heure car sinon hors des bornes de fin
        if rec["dateDebut"] <= ts.normalize() <= rec["dateFin"]:
            return rec["parti_recod"]
    return None


# application du match temporel
df["affiliation_mandat_députés"] = df.apply(get_parti_for_row, axis=1)

# Pas parfait mais pour avoir une idée :
print(
    "affectés :",
    df["affiliation_mandat_députés"].notna().sum(),
    "| non affectés :",
    df["affiliation_mandat_députés"].isna().sum(),
    "| id_acteur sans affiliation dynamique :",
    df[df["affiliation_mandat_députés"].isna()]["id_acteur"].nunique(),
)


affectés : 478064 | non affectés : 92995 | id_acteur sans affiliation dynamique : 204


### 2.2.3 Ajout des affiliations gouvernementales

### Création d'une variable sur-imprimant l'appartenance au gouv

In [14]:
# =================================================
# CRÉATION VARIABLE AFFILIATION + GOUV
# =================================================
# Ajout des affiliations gouvernementales
# renvoyer les membres du gouv à une catégorie "GOUV" pour les différencier
# le faire avant de forcer les groupeAbrev
# (qui feraient disparaître certains cas limites du gouvernement
# ie : si info affiliation manquante le même jour que d'autres affiliation GOUV)

"""
nb : traçabilité
/!\ ici on veut récup membres du gouv, souvent en sans affiliation
mais on veut aussi forcer leur etiquette gvt même quand ils ont une affiliation de député
(ex : ministre qui est aussi député)

Logique de recodage :
Recoder membres GVT, uniquement si != PA0 (= garder cohérence avec cas précédents)
si une des conditions suivantes est vérifiée,
- ministre -> ok, 96 personnes pour 130 qualité, mais exclure le cas de Justin Trudeau et 19 cas PA0
- garde des sceaux (pas toujours co-qualifié de ministre) : ok, 2 bien Dupond-Moretti / Belloubet (même si 10 PA0)
- secrétaire d’État -> 40 personnes pour 53 qualité correspondantes, OK (2 PA0)
= basé sur la lecture des résultats de :
df["qualite_orateur"].value_counts()

-> mais il faut exclure "Premier ministre du Canada" -> 2 occurences 
Autre option : exclure des PA PA-107309 = Justin Trudeau, Premier ministre du Canada
"""

# masque condition membres gouvernement
mask_gvt = (
    df["qualite_orateur"].str.contains(
        "ministre|garde des sceaux|secrétaire d[’']État",
        case=False,
        na=False,
        regex=True,
    )
    & (df["id_acteur"] != "PA0")
    & (df["id_acteur"] != "PA-107309")
)  # exclure Justin Trudeau, "Premier ministre du Canada"

# ========== Création nouvelle variable avec GVT ==========
# conserver l'affiliation initiale pour réutilisation future
df["affiliation_et_gouv"] = df["affiliation_mandat_députés"]
# recoder les cas concernés en GOUV
df.loc[mask_gvt, "affiliation_et_gouv"] = "GOUV"

# Vérification des cas affectés recodage GOUV
print("Lignes recodées GOUV :", mask_gvt.sum())
print(
    "Affiliation recodées pour",
    df.loc[mask_gvt, "id_acteur"].nunique(),
    "id_acteur uniques",
)

Lignes recodées GOUV : 82563
Affiliation recodées pour 109 id_acteur uniques


In [15]:
# ==========================================================
# RECODAGE CAS LIMITES GOUV :
# valeurs manquantes d'affiliation_et_gouv
# alors que valeurs GOUV le même jour
# ==========================================================

"""nb traçabilité :
Dans notre cas, après vérif manuelle, c'était des ratés
alors qu'on a bien une info gouv le même jour.
On a donc décidé de s'en tenir a cette version stricte.
/!\ D'autres pourraient envisager de recoder aussi les NA
entre une date connue avant et une date connue après,
même si pas le même jour (ex : NA entre 2 dates connues GOUV)
Ici la version conservatrice évite de bourrer des cas limites
de gens qui sortiraient puis reviendraient au gouv
sans que l'on prenne soin de vérifier les périodes.
"""

cols_audit = [
    "id_acteur",
    "nom_orateur_clean",
    "dateSeance_ts",
    "affiliation_et_gouv",
    "id_syceron",
]

tmp = df.loc[df["id_acteur"].notna() & (df["id_acteur"] != "PA0"), cols_audit].copy()
tmp = tmp.sort_values(["id_acteur", "dateSeance_ts"]).reset_index(drop=True)
g = tmp.groupby("id_acteur", group_keys=False)

# Valeurs non manquantes les plus proches avant/après
# contre inuitif les ffill et bfill mais c'est ça
tmp["prev_non_na_affil"] = g["affiliation_et_gouv"].ffill()
tmp["next_non_na_affil"] = g["affiliation_et_gouv"].bfill()

# Dates de référence (où affiliation_et_gouv est connue)
tmp["date_affil_connue"] = tmp["dateSeance_ts"].where(
    tmp["affiliation_et_gouv"].notna()
)
tmp["date_connue_avant"] = g["date_affil_connue"].ffill()
tmp["date_connue_apres"] = g["date_affil_connue"].bfill()

# NA entre deux bornes GOUV, même jour que l'une d'elles
# ie : même jour (avec dt.normalize() que la borne avant OU après
mask_a_recoder = (
    tmp["affiliation_et_gouv"].isna()
    & (tmp["prev_non_na_affil"] == "GOUV")
    & (tmp["next_non_na_affil"] == "GOUV")
    & (
        tmp["dateSeance_ts"].dt.normalize().eq(tmp["date_connue_avant"].dt.normalize())
        | tmp["dateSeance_ts"]
        .dt.normalize()
        .eq(tmp["date_connue_apres"].dt.normalize())
    )
)

# recodage dans df principal via id_syceron
ids_a_recoder = tmp.loc[mask_a_recoder, "id_syceron"]
mask_df = df["id_syceron"].isin(ids_a_recoder)
df.loc[mask_df, "affiliation_et_gouv"] = "GOUV"

# mini rapport post recodage
print(f"Lignes recodées GOUV (même jour) : {mask_df.sum()}")
print(f"id_syceron uniques : {df.loc[mask_df, 'id_syceron'].nunique()}")
print(f"id_acteur uniques : {df.loc[mask_df, 'id_acteur'].nunique()}")

print("\nPersonnes concernées :")
print(df.loc[mask_df, "nom_orateur_clean"].unique())

print("\nTop 10 orateurs recodés")
print(df.loc[mask_df, "nom_orateur_clean"].value_counts().head(10))

# 14 si sur affiliation après avoir forcé groupeAbrev
# 67 si fait avant (sur affiliation_mandat_députés)
# == FAIRE AVANT !!!

Lignes recodées GOUV (même jour) : 47
id_syceron uniques : 47
id_acteur uniques : 22

Personnes concernées :
['Mme Florence Parly' 'M. Olivier Dussopt' 'M. Julien Denormandie'
 'M. Bruno Le Maire' 'M. Adrien Taquet' 'M. Éric Dupond-Moretti'
 'M. Olivier Véran' 'Mme Élisabeth Borne' 'Mme Brigitte Klinkert'
 'Mme Frédérique Vidal' 'Mme Agnès Pannier-Runacher'
 'M. Christophe Castaner' 'Mme Brigitte Bourguignon' 'Mme Olivia Grégoire'
 'M. Gérald Darmanin' 'M. Jean-Noël Barrot' 'M. Thomas Cazenave'
 'Mme Aurore Bergé' 'Mme Fadila Khattabi' 'M. Roland Lescure'
 'M. Gabriel Attal' 'M. Marc Fesneau']

Top 10 orateurs recodés
nom_orateur_clean
M. Olivier Dussopt            6
M. Olivier Véran              5
M. Gérald Darmanin            5
M. Adrien Taquet              4
M. Éric Dupond-Moretti        4
Mme Olivia Grégoire           3
M. Marc Fesneau               2
M. Julien Denormandie         2
Mme Agnès Pannier-Runacher    2
M. Thomas Cazenave            2
Name: count, dtype: int64


### 2.2.4 Fallback des affiliations manquantes

#### Forcer le renvoi d'une affiliation si groupeAbrev connu

In [16]:
# ============================================================
# GESTION AFFILIATIONS MANQUANTES
# - Fallback pour les affiliations manquantes
# - Gestion des cas limites (RN, etc.)
# - TODO : Virer ou ajout d'affiliations après évaluation manuelle
# ============================================================


In [17]:
# ========= Fallback affiliation manquantes par groupeAbrev ==========

# TODO :avise si nécessaire désormais avec recodage gouv fait avant
# TODO : dans tous les cas garder le bloc pour ensuite sur-imprimer affil

# masque pour diagnostic des cas concernés par le fallback groupeAbrev
# (doit le placer avant de faire le combine_first pour avoir l'info)
mask_fallback = df["affiliation_et_gouv"].isna() & df["groupeAbrev"].notna()

# Forcer une affiliation avec le groupe "groupeAbrev" du fichier info députés
df["affiliation_et_gouv"] = df["affiliation_et_gouv"].combine_first(df["groupeAbrev"])
# réutiliser le même recodage que pour les affiliations
df["affiliation_et_gouv"] = df["affiliation_et_gouv"].replace(recodage_affiliation)
# Et gérer les nouvelles dénominations propres groupeAbrev
df["affiliation_et_gouv"] = df["affiliation_et_gouv"].replace(
    {"LES-REP": "LR", "UMP": "LR"}
)

print("=== Cas concernés par le fallback via groupeAbrev ===")
print("Nombre d'interventions concernées :", mask_fallback.sum())
print(
    "Nombre d'id_acteur uniques concernés :",
    df.loc[mask_fallback, "id_acteur"].nunique(dropna=True),
)

print("\nListe des orateurs concernés :")
print(df.loc[mask_fallback, "nom_orateur_clean"].dropna().unique())

# TODO : avant c'était des membres du gouv à qui on passait l'info avant identification gouv
# pour ceux qui avaient pas d'affiliation de mandat députés en cours
# désormais vérifier si vaut le coup car 3 interv qui sont peut être interrogés après leur mandat député ?
# = sont hors bornes mandats ou tout du moins appartenance groupe.


=== Cas concernés par le fallback via groupeAbrev ===
Nombre d'interventions concernées : 3
Nombre d'id_acteur uniques concernés : 3

Liste des orateurs concernés :
['Mme Christelle Dubos' 'Mme Valérie Boyer' 'M. Alain Bruneel']


#### Forcer affiliation des RN qui étaient en NI (étaient pas assez pour groupe)

In [18]:
# ========= Gestion cas limites RN ==========

# Recodage des RN de la XVe législature au bloc RN
# nb = choix = initialement en NI car pas assez nombreux pour former un groupe

liste_NI_RN = [
    "PA720822",  # Bruno Bilde
    "PA720668",  # Sébastien Chenu
    "PA720468",  # Emmanuel Blairy
    "PA720614",  # Marine Le Pen
    "PA719436",  # Nicolas Meizonnet
    "PA720802",  # Catherine Pujol
    "PA719608",  # Emmanuelle Ménard, rattachée au RN entre 2017 et 2022 mais plus entre 2022 et 2024
    "PA720606",  # Ludovic Pajot
    "PA606212",  # Gilbert Collard
    "PA720798",  # Louis Aliot
    "PA720610",  # Myriane Houplain, rattachée au RN entre 2017 et 2022, part ensuite reconquête ?
    # TODO : aviser du cas de Houplain que tu citais en todo mais pas ici
    # possible choix diff de ta part car elle part ensuite reconquête ?
    # mais si on applique même choix ça pourrait coller avec le fait
    # que pendant sa période d'intervention elle était rattachée au RN mais pas assez nb pour groupe ?
    # TODO : voir si d'autres cas NI/RN ? (cf multi affil)
]

# Date seuil : fin de la 15e législature
date_seuil = pd.Timestamp("2022-06-21")

# Condition combinée :
condition_NI_RN = (df["id_acteur"].isin(liste_NI_RN)) & (
    df["dateSeance_ts"].dt.normalize() < date_seuil
)  # dt.normalize() pour ignorer l'heure et éviter soucis de bornes


# Application de la modalité uniquement pour les lignes correspondant à la condition
df.loc[condition_NI_RN, "affiliation_et_gouv"] = "RN"

# Vérification
print("Lignes recodées RN :", condition_NI_RN.sum())
print(
    "Affiliation recodées pour",
    df.loc[condition_NI_RN, "id_acteur"].nunique(),
    "id_acteur uniques",
)

print("Ceci ne modifie pas nb sans affiliation_et_gouv : simple recodage NI vers RN")


Lignes recodées RN : 5648
Affiliation recodées pour 11 id_acteur uniques
Ceci ne modifie pas nb sans affiliation_et_gouv : simple recodage NI vers RN


In [19]:
# TODO : voir si nécessaire créer affiliation forcée
# si oui, écraser membre gouv par leur groupe abrev
# et forcer les autres cas limites
# Si on le fait :
# voir avec matthias si renvoi derniere affiliation suffit pour couleur politique globale.
# ou si on affine pour membre gouv qui étaient député y a longtemps
# (genre ici nous bachelot serait UMP -> avec un recod LR
# mais donc discutable et voir gestion manuelle membres gouv ?)

# GESTION DES CAS RESTANTS

In [20]:
# ==============================================
# TODO: AFFILIATIONS : MATTHIAS EN COURS
# explorer les affiliation manquantes pour identifier les cas limites
# ==============================================


In [21]:
# vérification des cas sans affiliation :
print(
    "Nombre restant d'interventions sans affiliation :",
    df["affiliation_et_gouv"].isna().sum(),
)
print(
    "Nombre restant d'id_acteur uniques ayant des affiliations manquantes :",
    df[df["affiliation_et_gouv"].isna()]["id_acteur"].nunique(),
)

Nombre restant d'interventions sans affiliation : 11761
Nombre restant d'id_acteur uniques ayant des affiliations manquantes : 96


In [22]:
# IDENTIFICATION CAS MANQUANTS ET LIMITES AFFILIATION ET GOUV

# Acteurs avec au moins un NA dans affiliation_et_gouv (hors PA0)
restant_affiliation_et_gouv = df[
    (df["affiliation_et_gouv"].isna()) & (df["id_acteur"] != "PA0")
]

# Comptage NA par acteur directement
count_restant_par_acteur = (
    restant_affiliation_et_gouv.groupby(
        ["id_acteur", "nom_orateur_clean"], dropna=False
    )
    .size()
    .reset_index(name="nb_na_interventions")
)

# Répartition NA / renseigné pour ces mêmes acteurs
repartition = (
    df[df["id_acteur"].isin(count_restant_par_acteur["id_acteur"])]
    .groupby("id_acteur")["affiliation_et_gouv"]
    .agg(
        nb_na=lambda s: s.isna().sum(),
        nb_renseigne=lambda s: s.notna().sum(),
    )
    .reset_index()
)

resultat = count_restant_par_acteur.merge(repartition, on="id_acteur").sort_values(
    "nb_renseigne", ascending=False
)

print(f"Nombre d'id_acteur avec au moins un NA : {resultat['id_acteur'].nunique()}")
display(resultat)

resultat.to_csv("../data/temp/count_restant_affiliation_et_gouv.csv", index=False)


Nombre d'id_acteur avec au moins un NA : 95


,id_acteur,nom_orateur_clean,nb_na_interventions,nb_na,nb_renseigne
46,PA-125309,Mme Myriam El Khomri,3,3,1
0,PA-103629,M. Gérard Larcher,1,1,0
60,PA-125479,Mme Sophie Vénétitay,8,8,0
69,PA-125569,M. Serge Durand,3,3,0
68,PA-125559,Mme Salomé Arbault,2,2,0
...,...,...,...,...,...
29,PA-122139,M. Daniel Salmon,2,2,0
28,PA-121669,M. Volodymyr Zelensky,2,2,0
27,PA-121659,M. Grégory Thuizat,3,3,0
26,PA-121649,M. Erwan Guermeur,4,4,0


In [23]:
# IDEM AVEC QUALITÉ ORATEUR

# IDENTIFICATION CAS MANQUANTS ET LIMITES AFFILIATION ET GOUV

# Acteurs avec au moins un NA dans affiliation_et_gouv (hors PA0)
restant_affiliation_et_gouv = df[
    (df["affiliation_et_gouv"].isna()) & (df["id_acteur"] != "PA0")
]

# Comptage NA par acteur directement
count_restant_par_acteur = (
    restant_affiliation_et_gouv.groupby(
        ["id_acteur", "nom_orateur_clean", "qualite_orateur"], dropna=False
    )
    .size()
    .reset_index(name="nb_na_interventions")
)

# Répartition NA / renseigné pour ces mêmes acteurs
repartition = (
    df[df["id_acteur"].isin(count_restant_par_acteur["id_acteur"])]
    .groupby("id_acteur")["affiliation_et_gouv"]
    .agg(
        nb_na=lambda s: s.isna().sum(),
        nb_renseigne=lambda s: s.notna().sum(),
    )
    .reset_index()
)

resultat = count_restant_par_acteur.merge(repartition, on="id_acteur").sort_values(
    "nb_renseigne", ascending=False
)

print(f"Nombre d'id_acteur avec au moins un NA : {resultat['id_acteur'].nunique()}")
display(resultat)

resultat.to_csv(
    "../data/temp/count_restant_qualite_affiliation_et_gouv.csv", index=False
)


Nombre d'id_acteur avec au moins un NA : 95


,id_acteur,nom_orateur_clean,qualite_orateur,nb_na_interventions,nb_na,nb_renseigne
72,PA-125309,Mme Myriam El Khomri,NaN,3,3,1
0,PA-103629,M. Gérard Larcher,président du Sénat,1,1,0
115,PA-125559,Mme Salomé Arbault,NaN,1,2,0
108,PA-125529,M. Loïg Chesnais-Girard,président de la région Bretagne et de la commi...,1,8,0
109,PA-125529,M. Loïg Chesnais-Girard,NaN,7,8,0
...,...,...,...,...,...,...
57,PA-125229,Mme Catherine Delgoulet,NaN,6,7,0
58,PA-125239,M. Didier Quercioli,secrétaire général de la Mutualité Fonction Pu...,1,7,0
59,PA-125239,M. Didier Quercioli,NaN,6,7,0
60,PA-125249,M. Erwan Lecœur,"sociologue, membre du laboratoire Pacte",1,10,0


##### LES SOUCIS POSSIBLES multi affil:


In [24]:
# Cas où un même id_acteur a plusieurs valeurs différentes de affiliation_et_gouv
tmp = df[["id_acteur", "nom_orateur_clean", "affiliation_et_gouv"]].copy()
tmp["affiliation_et_gouv_norm"] = tmp["affiliation_et_gouv"].fillna("<<NA>>")

# ids avec au moins 2 modalités différentes (en comptant NA)
ids_multi_affil = (
    tmp.groupby("id_acteur")["affiliation_et_gouv_norm"]
    .nunique()
    .loc[lambda s: s > 1]
    .index
)
cas_diff = tmp[tmp["id_acteur"].isin(ids_multi_affil)].copy()

print(
    "Nombre d'id_acteur avec plusieurs valeurs de affiliation_et_gouv :",
    len(ids_multi_affil),
)
print("Nombre total de lignes concernées :", len(cas_diff))

cas_multi_affil = (
    cas_diff.groupby(["id_acteur", "nom_orateur_clean"])["affiliation_et_gouv_norm"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="valeurs_affiliation_et_gouv")
    .sort_values(["nom_orateur_clean", "id_acteur"])
)
display(cas_multi_affil)
cas_multi_affil.to_csv("../data/temp/cas_multi_affiliation_et_gouv.csv", index=False)

Nombre d'id_acteur avec plusieurs valeurs de affiliation_et_gouv : 153
Nombre total de lignes concernées : 115496


,id_acteur,nom_orateur_clean,valeurs_affiliation_et_gouv
87,PA720422,M. Adrien Quatennens,"[LFI, NI]"
131,PA722086,M. Adrien Taquet,"[GOUV, REN]"
147,PA794914,M. Alexandre Vincendet,"[HOR, LR]"
25,PA421348,M. André Villiers,"[HOR, UDI]"
7,PA267355,M. Antoine Herth,"[AGIR-E, UDI]"
...,...,...,...
115,PA721514,Mme Stéphanie Kerbarh,"[LIOT, REN]"
20,PA336175,Mme Sylvia Pinel,"[LIOT, NI]"
89,PA720500,Mme Valérie Petit,"[AGIR-E, REN]"
57,PA719194,Mme Yolaine de Courson,"[DEM, EDS, NI, REN]"


##### LES SOUCIS POSSIBLES AVEC MEMBRES GOUV :

In [25]:
# CAS LIMITE GOUV :
# ORATEURS AVEC AFFIL = GOUV + AUTRE CHOSE : identification + comptage des interventions

# LOGIQUE :
# - RENVOYER LES CAS OU AFFIL = GOUV + AUTRE CHOSE
# - identification orateurs et combinaison d'affiliations
# - comptage des interventions

# Interventions des orateurs qui ont plusieurs affiliations dont GVT

# 1) Ne garder que les lignes avec affiliation renseignée
temp = df.loc[
    df["affiliation_et_gouv"].notna(),
    ["id_acteur", "nom_orateur_clean", "affiliation_et_gouv"],
].copy()
# (ça ici permet les cas sans affil forcée, mais marcherait aussi si on fait
# juste après l'affil dynamique pour pas rater les cas)
temp["affiliation_et_gouv_norm"] = temp["affiliation_et_gouv"].fillna("<<NA>>")

# 2) Profils d'affiliation par id_acteur → garder ceux avec affiliations multiples dont GOUV
affil_par_id = temp.groupby("id_acteur")["affiliation_et_gouv_norm"].agg(
    lambda s: sorted(set(s.astype(str)))
)
ids_multi_avec_gvt = affil_par_id[
    affil_par_id.apply(lambda x: len(x) > 1 and "GOUV" in x)
].index

# 3) Comptage GOUV vs AUTRE par orateur
subset = temp[temp["id_acteur"].isin(ids_multi_avec_gvt)].copy()
subset["type_intervention"] = (
    subset["affiliation_et_gouv"].eq("GOUV").map({True: "GOUV", False: "AUTRE"})
)

counts = subset["type_intervention"].value_counts()
nb_gouv = int(counts.get("GOUV", 0))
nb_autre = int(counts.get("AUTRE", 0))
print(f"Interventions comme membre du gouv : {nb_gouv}")
print(f"Interventions dans les autres cas  : {nb_autre}")
print(f"Total                              : {nb_gouv + nb_autre}")

# 4) Tableau fusionné : comptages + affiliations
par_orateur = (
    subset.groupby(["id_acteur", "nom_orateur_clean", "type_intervention"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
par_orateur["TOTAL"] = par_orateur.get("GOUV", 0) + par_orateur.get("AUTRE", 0)

affiliations = (
    subset.groupby(["id_acteur", "nom_orateur_clean"])["affiliation_et_gouv_norm"]
    .agg(lambda s: sorted(set(s.astype(str))))
    .reset_index(name="affiliations")
)

membres_gouv_multi_affil = par_orateur.merge(
    affiliations, on=["id_acteur", "nom_orateur_clean"]
).sort_values("TOTAL", ascending=False)
print(
    f"\nNombre d'orateurs avec affiliations multiples dont GOUV : {len(membres_gouv_multi_affil)}"
)
display(membres_gouv_multi_affil)

membres_gouv_multi_affil.to_csv(
    "../data/temp/membres_gouv_multi_affil.csv", index=False
)


Interventions comme membre du gouv : 28769
Interventions dans les autres cas  : 26954
Total                              : 55723

Nombre d'orateurs avec affiliations multiples dont GOUV : 48


,id_acteur,nom_orateur_clean,AUTRE,GOUV,TOTAL,affiliations
2,PA330357,M. Olivier Dussopt,106,5065,5171,"[GOUV, REN, SOC-A]"
0,PA267336,M. Joël Giraud,4300,178,4478,"[GOUV, REN]"
10,PA642788,M. Olivier Véran,1864,2568,4432,"[GOUV, REN]"
3,PA331582,M. Philippe Vigier,2566,138,2704,"[DEM, GOUV, LIOT, UDI]"
28,PA721134,M. Roland Lescure,1329,1164,2493,"[GOUV, REN]"
39,PA722190,M. Gabriel Attal,199,1928,2127,"[GOUV, REN]"
22,PA719938,M. Marc Fesneau,406,1657,2063,"[DEM, GOUV]"
38,PA722086,M. Adrien Taquet,66,1793,1859,"[GOUV, REN]"
16,PA719372,M. Guillaume Kasbarian,1706,101,1807,"[GOUV, REN]"
26,PA720512,M. Laurent Pietraszewski,628,1093,1721,"[GOUV, REN]"


In [26]:
# ==============================
# vérif et possibles soucis :
# ==============================

# TODO : cas limites breneel, boyer -> hors bornes, mais possible qu'ils reviennent comme intervenants externes en fait ?



In [27]:
# NOTE: Désormais géré directement depuis extraction
mask_congres = df["session"].str.contains("Congrès du Parlement", case=False, na=False)

print("Présence de 'Congrès du Parlement' dans session :", mask_congres.any())
print("Nombre de lignes concernées :", int(mask_congres.sum()))

# Optionnel : voir les valeurs de session concernées
if mask_congres.any():
    print("\nValeurs des sessions concernées :")
    print(df.loc[mask_congres, "session"].value_counts())  # Trucs Matthias


Présence de 'Congrès du Parlement' dans session : False
Nombre de lignes concernées : 0


In [28]:
# TODO : voir si jamais c'est justifié (reviendrait pas comme député ou gouv mais membre externe)

# # solution temporaire sur 2 cas étranges
# LM = OK QUAND ON FORCE LES AFFIL + pas identifié à cause bornes


# Boyer = ["PA330684"]  # cas similaire sur intervention du 7 novembre 2020
# LM explication : l'info d'affiliation s'arrête au 30 sept 2020
# "PM731296";"PA330684";"15";"2017-06-27";"2020-09-30";"20";"1";"Membre";"PO730934";"Les Républicains";"LR";"LR";"2017-06-27";"2022-06-21"

# df.loc[df["id_acteur"].isin(Boyer), "groupe&gvt_affiliation"] = "LR"

# Bruneel = [
#     "PA720546"
# ]  # ici cas étrange sur une intervention le 9 janvier 2023, il a été laissé en valeur manquante alors que GDR
# LM = l'info affiliation s'arrête au 21 juin 2022
# "PM731533";"PA720546";"15";"2017-06-27";"2022-06-21";"20";"1";"Membre";"PO730940";"Gauche démocrate et républicaine";"GDR";"GDR";"2017-06-27";"2022-06-21"

# df.loc[df["id_acteur"].isin(Bruneel), "groupe&gvt_affiliation"] = "GDR"

In [29]:
# TODO : on aura sans doute décidé plus haut, je garde au cas où pour l'instant
# TODO : léo, voir ces machins avec matthias ensuite pour clarifier.
# Et voir pourquoi passé par un isin plutôt que ==

# # Reste des cas particuliers à replacer dans leur affiliation au moment de leurs fonctions gouvernementales respectives
# Bachelot = ["PA332"]

# df.loc[df["id_acteur"].isin(Bachelot), "groupe_all_affiliation"] = (
#     "NI"  # NI ou mettre valeur manquante ? pareil pour Philippe, Le Drian, Rousseau
# )

# Vautrin = ["PA267797"]

# df.loc[df["id_acteur"].isin(Vautrin), "groupe_all_affiliation"] = "REN"

# Philippe = ["PA345619"]

# df.loc[df["id_acteur"].isin(Philippe), "groupe_all_affiliation"] = "NI"

# Ledrian = ["PA1872"]

# df.loc[df["id_acteur"].isin(Ledrian), "groupe_all_affiliation"] = "NI"

# Rousseau = ["PA826635"]

# df.loc[df["id_acteur"].isin(Rousseau), "groupe_all_affiliation"] = "NI"

## Export

In [30]:
# Export du csv nettoyé
df.to_csv("../data/interim/data_cleaning_full.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (cf : adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

# PROVISOIRE !! Regrouper les interventions interrompues NOT WORKING
ÇA NE MARCHE PAS POUR L'INSTANT !!!!!!


In [31]:
# TODO: À affiner et vérifier la fusion interventions interrompues
# FIXME: ça foire

# AVISER : pas le cas ici, mais envisager possible gestion des cas NaN
df_interruption = df[df["code_grammaire"].str.contains("INTERRUPTION")]
df_intervention = df[~df["code_grammaire"].str.contains("INTERRUPTION")]
# Si il fallait s'en assurer :
# is_interruption = df["code_grammaire"].str.contains("INTERRUPTION", na=False)
# df_interruption = df[is_interruption]
# df_intervention = df[~is_interruption]

# assert len(df_interruption) + len(df_intervention) == len(df), (
#     f"Lignes perdues lors du split ! "
#     f"{len(df)} ≠ {len(df_interruption)} + {len(df_intervention)} "
#     f"(NaN dans code_grammaire : {df['code_grammaire'].isna().sum()})"
# )

# TODO : NON ÇA VA PAS ÇA REGROUPE NAWAK ????

# ordinal_prise semble plus précis au niveau des intervenants
# = est constant quand interrompu là où les ordres obsolu et ptsodj changent
group_keys = ["uid", "dateSeance_ts", "id_acteur", "ordinal_prise"]

# agréger : concat texte, sommer longueur, garder premières infos utiles
agg = {
    "texte": lambda s: " ".join(s.dropna().astype(str)).strip(),
    "len_texte_brut": "sum",
    "code_parole": lambda s: ", ".join(sorted(set(s.dropna().astype(str)))),
    "id_syceron": lambda s: s.dropna().unique().tolist(),
    # "ordre_absolu_seance": "first", # list pour garder l'ordre des prises ?
    # "nom_orateur": "first",
    # "qualite_orateur": "first",
    # "id_orateur": "first",
    # "stime": "first",
}

# ajouter 'first' pour toutes les autres colonnes non clés/non déjà agrégées
for c in df_intervention.columns:
    if c not in group_keys and c not in agg:
        agg[c] = "first"

# Regroupe les interventions par clés communes et agrège les colonnes définies dans `agg`
df_intervention_grouped = (
    df_intervention.groupby(group_keys, dropna=False).agg(agg).reset_index()
)

# Recolle les interventions regroupées avec les interruptions
# puis aligne les colonnes sur le format d'origine
df_concat = pd.concat([df_intervention_grouped, df_interruption], ignore_index=True)[
    df_interruption.columns
]

# Retrier dans l'ordre chronologique et d'affichage de la séance
# nb : ici ok car gardé seulement first pour ordre_absolu_seance
# mais modif si jamais on avait gardé la liste complète des ordres
df_concat = df_concat.sort_values(
    by=["dateSeance_ts", "valeur_ptsodj", "ordre_absolu_seance"]
).reset_index(drop=True)

print(
    f"Regroupement des interventions interrompues \n"
    f"avant: {len(df)} | après: {len(df_concat)} "
    f"(interventions: de {len(df_intervention)} → à {len(df_intervention_grouped)}, "
    f"interruptions: {len(df_interruption)})"
)

# TODO : len_texte_brut sera à revoir pour interv groupée
# , car là on à la trace de la longueur des interventions avant regroupement
# puis on en fait une somme      "len_texte_brut": "sum",
# donc voir si clair pour une fois regroupé ?, ou si on recalcule plutôt une nouvelle var ?
# ie si peut porter à confusion

# Export du csv concat nettoyé
df_concat.to_csv("../data/interim/data_cleaning_grouped.csv", index=False)
print("Export du csv concat nettoyé : ../data/interim/data_cleaning_grouped.csv")
# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

Regroupement des interventions interrompues 
avant: 571059 | après: 423696 (interventions: de 338424 → à 191061, interruptions: 232635)
Export du csv concat nettoyé : ../data/interim/data_cleaning_grouped.csv


# EXPLORATION

In [32]:
# TODO : aller voir parce que ça regroupe quand meme des trucs qui
# ont pas le même code parole
# donc voir le pourquoi du comment
# MAIS ON S'EN COGNE UN PEU SUR LE PRINCIPE ?
# ENFIN AVISER QUE JUSTE LES AVIS GOUV SOIT PAS REGROUPÉS
# AVEC UNE PRISE PAROLE PLUS LARGE ?
# CHANGE RIEN DE DRAMATIQUE SANS DOUTE.

df_concat["code_parole"].value_counts()[:-10]

code_parole
non_précisé                                 287594
PAROLE_1_2                                   90506
PAROLE_1_2, non_précisé                      15004
AVIS_COM_1_20                                10835
AVIS_GVT_1_20                                10125
AVIS_GVT_1_20, PAROLE_1_2                     3334
AVIS_COM_1_20, non_précisé                    2265
AVIS_COM_1_20, PAROLE_1_2                     1769
AVIS_GVT_1_20, non_précisé                    1096
AVIS_COM_1_20, PAROLE_1_2, non_précisé         681
AVIS_GVT_1_20, PAROLE_1_2, non_précisé         452
AVIS_COM_1_20, AVIS_GVT_1_20                    14
AVIS_COM_1_20, AVIS_GVT_1_20, PAROLE_1_2         5
Name: count, dtype: int64

In [33]:
# ÇA REGROUPE NAWAK !!!!!

In [34]:
chelou = df_concat[df_concat["code_parole"] == "AVIS_GVT_1_20, PAROLE_1_2"]
chelou.head

<bound method NDFrame.head of                           uid               SeanceRef   SessionRef  \
387     CRSANR5L15S2017E1N003                    None         None   
424     CRSANR5L15S2017E1N004                    None         None   
770     CRSANR5L15S2017E1N006                    None         None   
850     CRSANR5L15S2017E1N007                    None         None   
1021    CRSANR5L15S2017E1N008                    None         None   
...                       ...                     ...          ...   
404535  CRSANR5L16S2024O1N150  RUANR5L16S2024IDS28146  SCR5A2024O1   
404721  CRSANR5L16S2024O1N151  RUANR5L16S2024IDS28218  SCR5A2024O1   
404754  CRSANR5L16S2024O1N151  RUANR5L16S2024IDS28218  SCR5A2024O1   
404822  CRSANR5L16S2024O1N152  RUANR5L16S2024IDS28157  SCR5A2024O1   
408634  CRSANR5L16S2024O1N170  RUANR5L16S2024IDS28206  SCR5A2024O1   

               dateSeance         dateSeanceJour numSeanceJour  numSeance  \
387     20170706093000000  jeudi 06 juillet 2017    

Je comprends bien l’intention exposée par M. Lagarde. Je rappelle toutefois l’existence de la disposition dont vient de parler M. le rapporteur.De surcroît, le contrôle des assemblées a été sensiblement renforcé lors de la quatrième prorogation de l’état d’urgence, en juillet 2016. Dès le 25 juillet 2016, les commissions des lois des deux assemblées se sont ainsi vu transmettre copie des mesures prises sur le fondement de la loi du 3 avril 1955, ce qui a permis aux rapporteurs concernés de disposer d’une connaissance exhaustive de toutes ces mesures. J’en ai parlé abondamment au Sénat, hier, avec M. le rapporteur Michel Mercier. Les rapporteurs des deux commissions des lois sont informés de tout ce qui se passe pendant l’état d’urgence. L’avis du Gouvernement est donc défavorable. Vous me permettrez d’exprimer mon accord avec M. Larrivé : les moyens de contrôle dont disposent aujourd’hui les commissions des lois de l’Assemblée nationale et du Sénat sont très importants. Les informations les plus confidentielles sont communiquées à leurs présidents et rapporteurs respectifs. Vous comprendrez aisément que, pendant la Guerre de Quatorze, le comité parlementaire était informé des grandes lignes stratégiques, pas forcément de la tactique déployée sur le terrain. L’avis du Gouvernement reste donc défavorable.

Je comprends bien l’intention exposée par M. Lagarde. Je rappelle toutefois l’existence de la disposition dont vient de parler M. le rapporteur.De surcroît, le contrôle des assemblées a été sensiblement renforcé lors de la quatrième prorogation de l’état d’urgence, en juillet 2016. Dès le 25 juillet 2016, les commissions des lois des deux assemblées se sont ainsi vu transmettre copie des mesures prises sur le fondement de la loi du 3 avril 1955, ce qui a permis aux rapporteurs concernés de disposer d’une connaissance exhaustive de toutes ces mesures. J’en ai parlé abondamment au Sénat, hier, avec M. le rapporteur Michel Mercier. Les rapporteurs des deux commissions des lois sont informés de tout ce qui se passe pendant l’état d’urgence. L’avis du Gouvernement est donc défavorable.

In [35]:
df[df["id_syceron"] == 983326][["nom_orateur_clean", "texte", "len_texte_brut"]]

,nom_orateur_clean,texte,len_texte_brut
352,M. Gérard Collomb,Je comprends bien l'intention exposée par M. L...,791.0


In [36]:
df[df["id_syceron"] == 983358]

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,job,nombreMandats,experienceDepute,scoreParticipation,scoreLoyaute,scoreMajorite,dateMaj,dateSeance_ts,affiliation_mandat_députés,affiliation_et_gouv
363,CRSANR5L15S2017E1N003,NaN,NaN,20170706093000000,jeudi 06 juillet 2017,1,3,AN,15,Première session extraordinaire 2017,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-07-06 09:30:00,None,GOUV


In [37]:
# Vérifier si code_parole varie au sein d'un même groupe
check = df_intervention.groupby(group_keys, dropna=False)["code_parole"].nunique()
print("Groupes avec code_parole non constant :", (check > 1).sum())

Groupes avec code_parole non constant : 24629


In [38]:
df["id_acteur"].str.contains("-").sum()

550

In [39]:
df_externe = df[df["id_acteur"].str.contains("-", regex=False, na=False)].copy()
df_externe

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,job,nombreMandats,experienceDepute,scoreParticipation,scoreLoyaute,scoreMajorite,dateMaj,dateSeance_ts,affiliation_mandat_députés,affiliation_et_gouv
63085,CRSANR5L15S2018O1N118,NaN,NaN,20180122170000000,lundi 22 janvier 2018,Unique,118,AN,15,Session ordinaire 2017-2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-01-22 17:00:00,None,NaN
66692,CRSANR5L15S2018O1N131,NaN,NaN,20180201093000000,jeudi 01 février 2018,1,131,AN,15,Session ordinaire 2017-2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-02-01 09:30:00,None,NaN
67642,CRSANR5L15S2018O1N136,NaN,NaN,20180207150000000,mercredi 07 février 2018,1,136,AN,15,Session ordinaire 2017-2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-02-07 15:00:00,None,NaN
67644,CRSANR5L15S2018O1N136,NaN,NaN,20180207150000000,mercredi 07 février 2018,1,136,AN,15,Session ordinaire 2017-2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-02-07 15:00:00,None,NaN
79718,CRSANR5L15S2018O1N192,NaN,NaN,20180417150000000,mardi 17 avril 2018,2,192,AN,15,Session ordinaire 2017-2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-04-17 15:00:00,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
555690,CRSANR5L16S2024O1N184,RUANR5L16S2024IDS28279,SCR5A2024O1,20240506150000000,lundi 06 mai 2024,1,184,AN,16,Session ordinaire 2023-2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-06 15:00:00,None,NaN
555691,CRSANR5L16S2024O1N184,RUANR5L16S2024IDS28279,SCR5A2024O1,20240506150000000,lundi 06 mai 2024,1,184,AN,16,Session ordinaire 2023-2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-06 15:00:00,None,NaN
555692,CRSANR5L16S2024O1N184,RUANR5L16S2024IDS28279,SCR5A2024O1,20240506150000000,lundi 06 mai 2024,1,184,AN,16,Session ordinaire 2023-2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-06 15:00:00,None,NaN
555694,CRSANR5L16S2024O1N184,RUANR5L16S2024IDS28279,SCR5A2024O1,20240506150000000,lundi 06 mai 2024,1,184,AN,16,Session ordinaire 2023-2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-06 15:00:00,None,NaN


In [40]:
resume_externes = (
    df_externe.groupby("id_acteur", dropna=False)
    .agg(
        id_orateur=(
            "id_orateur",
            lambda s: ", ".join(sorted(set(s.dropna().astype(str)))),
        ),
        nom_orateur=(
            "nom_orateur",
            lambda s: ", ".join(sorted(set(s.dropna().astype(str)))),
        ),
        nom_orateur_clean=(
            "nom_orateur_clean",
            lambda s: ", ".join(sorted(set(s.dropna().astype(str)))),
        ),
        qualite_orateur=(
            "qualite_orateur",
            lambda s: ", ".join(sorted(set(s.dropna().astype(str)))),
        ),
        nb_interventions=("id_syceron", "count"),
    )
    .reset_index()
    .sort_values("nb_interventions", ascending=False)
)

display(resume_externes)

,id_acteur,id_orateur,nom_orateur,nom_orateur_clean,qualite_orateur,nb_interventions
6,PA-121339,PA-121339,Mme Olivia Grégoire,Mme Olivia Grégoire,présidente de la commission spéciale,37
75,PA-125629,PA-125629,"M. Lyes Louffok, M. Lyes Louffok,",M. Lyes Louffok,"militant des droits de l’enfant, président de ...",21
14,PA-121449,PA-121449,M. Pierre Moscovici,M. Pierre Moscovici,"Premier président de la Cour des comptes, prem...",17
1,PA-103679,PA-103679,M. Didier Migaud,M. Didier Migaud,Premier président de la Cour des comptes,14
92,PA-1290,,"M. Jean-Paul Hamon, Mme Anne Souyris, Mme Laet...",Mme Laetitia Buffet,,12
...,...,...,...,...,...,...
3,PA-107299,PA-107299,M. Wolfgang Schäuble,M. Wolfgang Schäuble,président du Bundestag,1
30,PA-122409,PA-122409,Mme Sigrid Gérardin,Mme Sigrid Gérardin,cosecrétaire générale du SNUEP-FSU,1
31,PA-124459,PA-124459,Mme Sigrid Gérardin,Mme Sigrid Gérardin,,1
33,PA-125159,PA-125159,M. Rouslan Stefantchouk,M. Rouslan Stefantchouk,président de la Rada de l’Ukraine,1


In [41]:
df[
    (df["id_mandat"] == "-1")
    & (df["affiliation_et_gouv"] != "GOUV")
    & (df["id_acteur"] != "PA0")
]["nom_orateur_clean"].value_counts().head(50)

nom_orateur_clean
M. Jean-Paul Delevoye          54
Mme Olivia Grégoire            37
M. Lyes Louffok                21
M. Pierre Moscovici            17
Mme Laetitia Buffet            16
M. Didier Migaud               14
M. Manuel Domergue             12
Mme Lina Chamlal               12
M. Vincent Ploquin             10
M. Erwan Lecœur                10
M. Philippe Chalmin            10
M. Pierre-Alain Sarthou        10
Mme Noémie Ninnin               9
Mme Sophie Vénétitay            8
M. Fabrice Lenglart             8
Mme Diodio Metro                8
M. Christophe Carval            8
M. Patrick Weil                 8
M. Loïg Chesnais-Girard         8
M. Janmari Flower               7
Mme Catherine Perret            7
M. Martial Crance               7
M. Patrick Baudouin             7
Mme Sophie Taillé-Polian        7
M. Boris Vallaud                7
M. Yvon Serenus                 7
M. Philippe Pierre-Charles      7
M. Henri Sterdyniak             7
M. Sébastien Philippe         

In [42]:
df[  # (df["id_mandat"] == "-1") &
    (df["affiliation_et_gouv"] != "GOUV")
    & (df["id_acteur"] != "PA0")
    & (df["affiliation_et_gouv"].isna())
]["nom_orateur_clean"].value_counts().head(20)

nom_orateur_clean
M. Jean-Paul Delevoye      54
Mme Olivia Grégoire        37
M. Lyes Louffok            21
M. Pierre Moscovici        17
Mme Laetitia Buffet        16
M. Didier Migaud           14
M. Manuel Domergue         12
Mme Lina Chamlal           12
M. Vincent Ploquin         10
M. Erwan Lecœur            10
M. Philippe Chalmin        10
M. Pierre-Alain Sarthou    10
Mme Noémie Ninnin           9
Mme Sophie Vénétitay        8
M. Fabrice Lenglart         8
Mme Diodio Metro            8
M. Christophe Carval        8
M. Patrick Weil             8
M. Loïg Chesnais-Girard     8
M. Janmari Flower           7
Name: count, dtype: int64